# Titanic competition with TensorFlow Decision Forests

This notebook will take you through the steps needed to train a baseline Gradient Boosted Trees Model using TensorFlow Decision Forests and creating a submission on the Titanic competition. 

This notebook shows:

1. How to do some basic pre-processing. For example, the passenger names will be tokenized, and ticket names will be splitted in parts.
1. How to train a Gradient Boosted Trees (GBT) with default parameters
1. How to train a GBT with improved default parameters
1. How to tune the parameters of a GBTs
1. How to train and ensemble many GBTs

# Imports dependencies

In [1]:
import numpy as np
import pandas as pd
import os

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print(f"Found TF-DF {tfdf.__version__}")

Found TF-DF 1.2.0


# Load dataset

In [2]:
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
serving_df = pd.read_csv("/kaggle/input/titanic/test.csv")

train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [3]:
#欠損値の確認
train_df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
#最頻値で埋める
train_df["Age"] = train_df["Age"].fillna(train_df["Age"].mode()[0])
train_df["Cabin"] = train_df["Cabin"].fillna(train_df["Cabin"].mode()[0])
train_df["Embarked"] = train_df["Embarked"].fillna(train_df["Embarked"].mode()[0])
#再度確認
train_df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

In [5]:
#性別を数字にする
sex_map = {'male': 0, 'female': 1}
train_df['Sex'] = train_df['Sex'].map(sex_map)
train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,B96 B98,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,B96 B98,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,B96 B98,S
5,6,0,3,"Moran, Mr. James",0,24.0,0,0,330877,8.4583,B96 B98,Q
6,7,0,1,"McCarthy, Mr. Timothy J",0,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",0,2.0,3,1,349909,21.0750,B96 B98,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",1,27.0,0,2,347742,11.1333,B96 B98,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",1,14.0,1,0,237736,30.0708,B96 B98,C


In [6]:
# Embarked列をOne-Hotエンコーディングします
#embarked_dummies = pd.get_dummies(train_df['Embarked'], prefix='Embarked')

# 元のデータ表に、新しく作成した3つの列を連結します
#train_df = pd.concat([train_df, embarked_dummies], axis=1)

# 不要になった元のEmbarked列は削除します
#train_df = train_df.drop('Embarked', axis=1)

# 変換後のデータを確認します
#train_df.head(10)

In [7]:
# 数値データに関する基本的な統計量を表示します
print(train_df.describe())

       PassengerId    Survived      Pclass         Sex         Age  \
count   891.000000  891.000000  891.000000  891.000000  891.000000   
mean    446.000000    0.383838    2.308642    0.352413   28.566970   
std     257.353842    0.486592    0.836071    0.477990   13.199572   
min       1.000000    0.000000    1.000000    0.000000    0.420000   
25%     223.500000    0.000000    2.000000    0.000000   22.000000   
50%     446.000000    0.000000    3.000000    0.000000   24.000000   
75%     668.500000    1.000000    3.000000    1.000000   35.000000   
max     891.000000    1.000000    3.000000    1.000000   80.000000   

            SibSp       Parch        Fare  
count  891.000000  891.000000  891.000000  
mean     0.523008    0.381594   32.204208  
std      1.102743    0.806057   49.693429  
min      0.000000    0.000000    0.000000  
25%      0.000000    0.000000    7.910400  
50%      0.000000    0.000000   14.454200  
75%      1.000000    0.000000   31.000000  
max      8.000000

### データの規模（count）
全体のデータ数は 891行 あり、それぞれの項目（PassengerId、Survived、Pclassなど）が揃っていることが分かります。
### 生存率（Survived）
Survived 列の平均値（mean）が 約0.384（38.4%） です。これは、訓練データ全体の中で生存した人の割合が約38%であったことを示しています。
### 乗客の階級（Pclass）
Pclass（チケットクラス）の最小値が1、最大値が3、平均値が 約2.3 です。全体の分布として、1等客室よりも3等客室（下位クラス）の乗客の割合が多い傾向が読み取れます。
### 年齢の傾向（Age）
Age の平均は約28.6歳、中央値（50%）は24歳、最大値は80歳、最小値は約0.5歳（生後数ヶ月の乳児）となっています。幅広い年齢層の乗客が乗船していたことがわかります。
### 家族・同伴者の状況（SibSp / Parch）
SibSp（兄弟・配偶者の数）や Parch（両親・子供の数）の平均はいずれも0.3〜0.5付近ですが、最大値を見ると SibSp は8、Parch は6となっています。大部分の人は一人または少人数で乗船している一方で、大家族で乗っている人も一部に存在することがわかります。
### 運賃のばらつき（Fare）
Fare（運賃）の平均は約32.2ドルですが、中央値（50%）は14.5ドル、そして最大値は 512.3ドル に達しています。大部分の人は比較的安価な切符で乗っているのに対し、一部の非常に高額な切符を買った富裕層が存在するため、データが大きく右に歪んでいる（一部の超高額な外れ値がある）ことが読み取れます。

In [8]:
serving_df.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [9]:
#最頻値で埋める
serving_df["Age"] = serving_df["Age"].fillna(serving_df["Age"].mode()[0])
serving_df["Fare"] = serving_df["Fare"].fillna(serving_df["Fare"].mode()[0])
serving_df["Cabin"] = serving_df["Cabin"].fillna(serving_df["Cabin"].mode()[0])
#再度確認
serving_df.isnull().sum()

PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

# Prepare dataset

We will apply the following transformations on the dataset.

1. Tokenize the names. For example, "Braund, Mr. Owen Harris" will become ["Braund", "Mr.", "Owen", "Harris"].
2. Extract any prefix in the ticket. For example ticket "STON/O2. 3101282" will become "STON/O2." and 3101282.

In [10]:
def preprocess(df):
    df = df.copy()
    
    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])
    
    def ticket_number(x):
        return x.split(" ")[-1]
        
    def get_title(name):
        title = name.split(",")[1].split(".")[0].strip()
        rare_titles = ["Lady","Countess","Capt","Col","Don","Dr",
                       "Major","Rev","Sir","Jonkheer","Dona"]
        if title in rare_titles:
            return "Rare"
        mapping = {"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"}
        return mapping.get(title, title)
    
    df["Title"] = df["Name"].apply(get_title)
    
    # Sex と Pclass を文字列として結合してから Sex を数値にマッピングする
    df["Sex_Pclass"] = df["Sex"].astype(str) + "_" + df["Pclass"].astype(str)
    
    
    df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    
    return df

# データの読み込み直後（欠損値埋めの後など）に、そのまま preprocess を呼ぶ
preprocessed_train_df = preprocess(train_df)
preprocessed_serving_df = preprocess(serving_df)

preprocessed_train_df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Sex_Pclass,Ticket_number
0,1,0,3,Braund Mr Owen Harris,0,22.0,1,0,A/5 21171,7.2500,B96 B98,S,Mr,0_3,21171
1,2,1,1,Cumings Mrs John Bradley Florence Briggs Thayer,1,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,1_1,17599
2,3,1,3,Heikkinen Miss Laina,1,26.0,0,0,STON/O2. 3101282,7.9250,B96 B98,S,Miss,1_3,3101282
3,4,1,1,Futrelle Mrs Jacques Heath Lily May Peel,1,35.0,1,0,113803,53.1000,C123,S,Mrs,1_1,113803
4,5,0,3,Allen Mr William Henry,0,35.0,0,0,373450,8.0500,B96 B98,S,Mr,0_3,373450


Let's keep the list of the input features of the model. Notably, we don't want to train our model on the "PassengerId" and "Ticket" features.

In [11]:
input_features = list(preprocessed_train_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
input_features.remove("Survived")
#input_features.remove("Ticket_number")
#input_features.remove("Embarked")     
input_features.remove("Cabin")  
input_features.remove("Sex")  
input_features.remove("Pclass")  
input_features.remove("SibSp")  
input_features.remove("Parch")  
input_features.remove("Fare") 
input_features.remove("Name")

print(f"Input features: {input_features}")

Input features: ['Age', 'Embarked', 'Title', 'Sex_Pclass', 'Ticket_number']


# Convert Pandas dataset to TensorFlow Dataset

In [12]:
def tokenize_names(features, labels=None):
    """Divite the names into tokens. TF-DF can consume text tokens natively."""
    features["Name"] =  tf.strings.split(features["Name"])
    return features, labels

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_train_df,label="Survived").map(tokenize_names)
serving_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_serving_df).map(tokenize_names)

# Train model with default parameters

### Train model

First, we are training a GradientBoostedTreesModel model with the default parameters.

In [13]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=2, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    random_seed=1234,
)
model.compile(metrics=["accuracy"])
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

Use 4 thread(s) for training
Use /tmp/tmpkzgg2r0p as temporary training directory
Reading training dataset...
Training tensor examples:
Features: {'PassengerId': <tf.Tensor 'data_7:0' shape=(None,) dtype=int64>, 'Pclass': <tf.Tensor 'data_8:0' shape=(None,) dtype=int64>, 'Name': tf.RaggedTensor(values=Tensor("data_4:0", shape=(None,), dtype=string), row_splits=Tensor("data_5:0", shape=(None,), dtype=int64)), 'Sex': <tf.Tensor 'data_9:0' shape=(None,) dtype=int64>, 'Age': <tf.Tensor 'data:0' shape=(None,) dtype=float64>, 'SibSp': <tf.Tensor 'data_11:0' shape=(None,) dtype=int64>, 'Parch': <tf.Tensor 'data_6:0' shape=(None,) dtype=int64>, 'Ticket': <tf.Tensor 'data_12:0' shape=(None,) dtype=string>, 'Fare': <tf.Tensor 'data_3:0' shape=(None,) dtype=float64>, 'Cabin': <tf.Tensor 'data_1:0' shape=(None,) dtype=string>, 'Embarked': <tf.Tensor 'data_2:0' shape=(None,) dtype=string>, 'Title': <tf.Tensor 'data_14:0' shape=(None,) dtype=string>, 'Sex_Pclass': <tf.Tensor 'data_10:0' shape=(None,

[INFO 2026-09-06T06:33:24.708644235+00:00 kernel.cc:756] Start Yggdrasil model training
[INFO 2026-09-06T06:33:24.709499901+00:00 kernel.cc:757] Collect training examples
[INFO 2026-09-06T06:33:24.711841681+00:00 kernel.cc:388] Number of batches: 1
[INFO 2026-09-06T06:33:24.711871953+00:00 kernel.cc:389] Number of examples: 891
[INFO 2026-09-06T06:33:24.712755325+00:00 data_spec_inference.cc:303] 671 item(s) have been pruned (i.e. they are considered out of dictionary) for the column Ticket_number (8 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
[INFO 2026-09-06T06:33:24.71300654+00:00 data_spec_inference.cc:303] 1 item(s) have been pruned (i.e. they are considered out of dictionary) for the column Title (5 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
[INFO 2026-09-06T06:33:24.713205811+00:00 kernel.cc:774] Training dataset:
Number of records: 891
Number of columns: 6

Number of columns by type:
	CATEGORICAL: 5 (83.3333%)
	

Model trained in 0:00:00.221097
Compiling model...
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Model compiled.
Accuracy: 0.8260869383811951 Loss:0.8624987602233887


# Train model with improved default parameters

Now you'll use some specific parameters when creating the GBT model

In [14]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=2, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    
    #num_trees=2000,
    
    # Only for GBT.
    # A bit slower, but great to understand the model.
    # compute_permutation_variable_importance=True,
    
    # Change the default hyper-parameters
    # hyperparameter_template="benchmark_rank1@v1",
    
    #num_trees=1000,
    #tuner=tuner
    
    min_examples=10,
    categorical_algorithm="RANDOM",
    max_depth=5,
    shrinkage=0.05,
    #num_candidate_attributes_ratio=0.2,
    split_axis="SPARSE_OBLIQUE",
    sparse_oblique_normalization="MIN_MAX",
    sparse_oblique_num_projections_exponent=2.0,
    num_trees=2000,
    #validation_ratio=0.0,
    random_seed=1234,
    
)
model.compile(metrics=["accuracy"])
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

Use 4 thread(s) for training
Use /tmp/tmpsf3dybr6 as temporary training directory
Reading training dataset...
Training tensor examples:
Features: {'PassengerId': <tf.Tensor 'data_7:0' shape=(None,) dtype=int64>, 'Pclass': <tf.Tensor 'data_8:0' shape=(None,) dtype=int64>, 'Name': tf.RaggedTensor(values=Tensor("data_4:0", shape=(None,), dtype=string), row_splits=Tensor("data_5:0", shape=(None,), dtype=int64)), 'Sex': <tf.Tensor 'data_9:0' shape=(None,) dtype=int64>, 'Age': <tf.Tensor 'data:0' shape=(None,) dtype=float64>, 'SibSp': <tf.Tensor 'data_11:0' shape=(None,) dtype=int64>, 'Parch': <tf.Tensor 'data_6:0' shape=(None,) dtype=int64>, 'Ticket': <tf.Tensor 'data_12:0' shape=(None,) dtype=string>, 'Fare': <tf.Tensor 'data_3:0' shape=(None,) dtype=float64>, 'Cabin': <tf.Tensor 'data_1:0' shape=(None,) dtype=string>, 'Embarked': <tf.Tensor 'data_2:0' shape=(None,) dtype=string>, 'Title': <tf.Tensor 'data_14:0' shape=(None,) dtype=string>, 'Sex_Pclass': <tf.Tensor 'data_10:0' shape=(None,

[INFO 2026-09-06T06:33:27.312438533+00:00 kernel.cc:756] Start Yggdrasil model training
[INFO 2026-09-06T06:33:27.31247192+00:00 kernel.cc:757] Collect training examples
[INFO 2026-09-06T06:33:27.312605202+00:00 kernel.cc:388] Number of batches: 1
[INFO 2026-09-06T06:33:27.312621919+00:00 kernel.cc:389] Number of examples: 891
[INFO 2026-09-06T06:33:27.313446476+00:00 data_spec_inference.cc:303] 671 item(s) have been pruned (i.e. they are considered out of dictionary) for the column Ticket_number (8 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
[INFO 2026-09-06T06:33:27.313588196+00:00 data_spec_inference.cc:303] 1 item(s) have been pruned (i.e. they are considered out of dictionary) for the column Title (5 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
[INFO 2026-09-06T06:33:27.313751272+00:00 kernel.cc:774] Training dataset:
Number of records: 891
Number of columns: 6

Number of columns by type:
	CATEGORICAL: 5 (83.3333%)
	

Model trained in 0:00:00.338107
Compiling model...
Model compiled.
Accuracy: 0.8260869383811951 Loss:0.856290876865387


Let's look at the model and you can also notice the information about variable importance that the model figured out

In [15]:
model.summary()

Model: "gradient_boosted_trees_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1
Trainable params: 0
Non-trainable params: 1
_________________________________________________________________
Type: "GRADIENT_BOOSTED_TREES"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (5):
	Age
	Embarked
	Sex_Pclass
	Ticket_number
	Title

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1.    "Sex_Pclass"  0.469417 ################
    2.           "Age"  0.384806 ##########
    3. "Ticket_number"  0.284567 ###
    4.         "Title"  0.277402 ##
    5.      "Embarked"  0.240010 

Variable Importance: NUM_AS_ROOT:
    1.    "Sex_Pclass" 76.000000 ################
    2. "Ticket_number" 30.000000 #####
    3.           "Age" 12.000000 #
    4.         "Title"  6.000000 
    5.      "Embarked"  3.000000 

Variable Importance: NUM_NODES:
    1.           "Age" 658.000000 ##############

In [16]:
model_with_importance = tfdf.keras.GradientBoostedTreesModel(
    verbose=2,
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True,
    compute_permutation_variable_importance=True,
    random_seed=1234,
)
model_with_importance.fit(train_ds)
model_with_importance.make_inspector().variable_importances()

Use 4 thread(s) for training
Use /tmp/tmp02rsczkr as temporary training directory
Reading training dataset...
Training tensor examples:
Features: {'PassengerId': <tf.Tensor 'data_7:0' shape=(None,) dtype=int64>, 'Pclass': <tf.Tensor 'data_8:0' shape=(None,) dtype=int64>, 'Name': tf.RaggedTensor(values=Tensor("data_4:0", shape=(None,), dtype=string), row_splits=Tensor("data_5:0", shape=(None,), dtype=int64)), 'Sex': <tf.Tensor 'data_9:0' shape=(None,) dtype=int64>, 'Age': <tf.Tensor 'data:0' shape=(None,) dtype=float64>, 'SibSp': <tf.Tensor 'data_11:0' shape=(None,) dtype=int64>, 'Parch': <tf.Tensor 'data_6:0' shape=(None,) dtype=int64>, 'Ticket': <tf.Tensor 'data_12:0' shape=(None,) dtype=string>, 'Fare': <tf.Tensor 'data_3:0' shape=(None,) dtype=float64>, 'Cabin': <tf.Tensor 'data_1:0' shape=(None,) dtype=string>, 'Embarked': <tf.Tensor 'data_2:0' shape=(None,) dtype=string>, 'Title': <tf.Tensor 'data_14:0' shape=(None,) dtype=string>, 'Sex_Pclass': <tf.Tensor 'data_10:0' shape=(None,

[INFO 2026-09-06T06:33:28.215638995+00:00 kernel.cc:756] Start Yggdrasil model training
[INFO 2026-09-06T06:33:28.215680939+00:00 kernel.cc:757] Collect training examples
[INFO 2026-09-06T06:33:28.215863649+00:00 kernel.cc:388] Number of batches: 1
[INFO 2026-09-06T06:33:28.215887021+00:00 kernel.cc:389] Number of examples: 891
[INFO 2026-09-06T06:33:28.216567814+00:00 data_spec_inference.cc:303] 671 item(s) have been pruned (i.e. they are considered out of dictionary) for the column Ticket_number (8 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
[INFO 2026-09-06T06:33:28.21675353+00:00 data_spec_inference.cc:303] 1 item(s) have been pruned (i.e. they are considered out of dictionary) for the column Title (5 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
[INFO 2026-09-06T06:33:28.216974748+00:00 kernel.cc:774] Training dataset:
Number of records: 891
Number of columns: 6

Number of columns by type:
	CATEGORICAL: 5 (83.3333%)
	

Model trained in 0:00:00.185377
Compiling model...
Model compiled.


{'MEAN_DECREASE_IN_AP_2_VS_OTHERS': [("Sex_Pclass" (4; #2),
   0.2808977330164184),
  ("Title" (4; #4), 0.12049825638550782),
  ("Embarked" (4; #1), 0.02298440698124604),
  ("Ticket_number" (4; #3), 0.021366087742330286),
  ("Age" (1; #0), -0.018132492061678218)],
 'NUM_AS_ROOT': [("Sex_Pclass" (4; #2), 32.0),
  ("Title" (4; #4), 5.0),
  ("Ticket_number" (4; #3), 1.0)],
 'INV_MEAN_MIN_DEPTH': [("Sex_Pclass" (4; #2), 0.686364276999472),
  ("Age" (1; #0), 0.3394611720564485),
  ("Title" (4; #4), 0.2674193571283735),
  ("Ticket_number" (4; #3), 0.20867605370346096),
  ("Embarked" (4; #1), 0.19855359425360083)],
 'MEAN_DECREASE_IN_ACCURACY': [("Sex_Pclass" (4; #2), 0.15217387676239014),
  ("Title" (4; #4), 0.1086956262588501),
  ("Embarked" (4; #1), 0.03260868787765503),
  ("Age" (1; #0), 0.02173912525177002),
  ("Ticket_number" (4; #3), 0.02173912525177002)],
 'NUM_NODES': [("Age" (1; #0), 449.0),
  ("Sex_Pclass" (4; #2), 106.0),
  ("Title" (4; #4), 76.0),
  ("Embarked" (4; #1), 66.0),
  

# Make predictions

In [17]:
def prediction_to_kaggle_format(model, threshold=0.5):
    proba_survive = model.predict(serving_ds, verbose=0)[:,0]
    return pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (proba_survive >= threshold).astype(int)
    })

def make_submission(kaggle_predictions):
    path="/kaggle/working/submission.csv"
    kaggle_predictions.to_csv(path, index=False)
    print(f"Submission exported to {path}")
    
kaggle_predictions = prediction_to_kaggle_format(model)
make_submission(kaggle_predictions)
!head /kaggle/working/submission.csv

Submission exported to /kaggle/working/submission.csv
PassengerId,Survived
892,1
893,1
894,1
895,1
896,1
897,1
898,1
899,0
900,1


# Training a model with hyperparameter tunning

Hyper-parameter tuning is enabled by specifying the tuner constructor argument of the model. The tuner object contains all the configuration of the tuner (search space, optimizer, trial and objective).


In [18]:
tuner = tfdf.tuner.RandomSearch(num_trials=1000)
tuner.choice("min_examples", [2, 5, 7, 10])
tuner.choice("categorical_algorithm", ["CART", "RANDOM"])

local_search_space = tuner.choice("growing_strategy", ["LOCAL"])
local_search_space.choice("max_depth", [3, 4, 5, 6, 8])

global_search_space = tuner.choice("growing_strategy", ["BEST_FIRST_GLOBAL"], merge=True)
global_search_space.choice("max_num_nodes", [16, 32, 64, 128, 256])

#tuner.choice("use_hessian_gain", [True, False])
tuner.choice("shrinkage", [0.02, 0.05, 0.10, 0.15])
tuner.choice("num_candidate_attributes_ratio", [0.2, 0.5, 0.9, 1.0])


tuner.choice("split_axis", ["AXIS_ALIGNED"])
oblique_space = tuner.choice("split_axis", ["SPARSE_OBLIQUE"], merge=True)
oblique_space.choice("sparse_oblique_normalization",
                     ["NONE", "STANDARD_DEVIATION", "MIN_MAX"])
oblique_space.choice("sparse_oblique_weights", ["BINARY", "CONTINUOUS"])
oblique_space.choice("sparse_oblique_num_projections_exponent", [1.0, 1.5])

# Tune the model. Notice the `tuner=tuner`.
tuned_model = tfdf.keras.GradientBoostedTreesModel(tuner=tuner)
tuned_model.fit(train_ds, verbose=0)

tuned_self_evaluation = tuned_model.make_inspector().evaluation()
print(f"Accuracy: {tuned_self_evaluation.accuracy} Loss:{tuned_self_evaluation.loss}")

Use /tmp/tmpmom6qc3b as temporary training directory


[INFO 2026-09-06T06:35:25.623861543+00:00 kernel.cc:1214] Loading model from path /tmp/tmpmom6qc3b/model/ with prefix bf9cf61ab9684def
[INFO 2026-09-06T06:35:25.636581233+00:00 decision_forest.cc:661] Model loaded with 24 root(s), 744 node(s), and 14 input feature(s).
[INFO 2026-09-06T06:35:25.636624203+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesGeneric" built
[INFO 2026-09-06T06:35:25.636648202+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.8767123222351074 Loss:0.6209227442741394


In the last line in the cell above, you can see the accuracy is higher than previously with default parameters and parameters set by hand.

This is the main idea behing hyperparameter tuning.

For more information you can follow this tutorial: [Automated hyper-parameter tuning](https://www.tensorflow.org/decision_forests/tutorials/automatic_tuning_colab)

# Making an ensemble

Here you'll create 100 models with different seeds and combine their results

This approach removes a little bit the random aspects related to creating ML models

In the GBT creation is used the `honest` parameter. It will use different training examples to infer the structure and the leaf values. This regularization technique trades examples for bias estimates.

In [19]:
predictions = None
num_predictions = 0

for i in range(100):
    print(f"i:{i}")
    # Possible models: GradientBoostedTreesModel or RandomForestModel
    model = tfdf.keras.GradientBoostedTreesModel(
        verbose=0, # Very few logs
        features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
        exclude_non_specified_features=True, # Only use the features in "features"

        #min_examples=1,
        #categorical_algorithm="RANDOM",
        ##max_depth=4,
        #shrinkage=0.05,
        ##num_candidate_attributes_ratio=0.2,
        #split_axis="SPARSE_OBLIQUE",
        #sparse_oblique_normalization="MIN_MAX",
        #sparse_oblique_num_projections_exponent=2.0,
        #num_trees=2000,
        ##validation_ratio=0.0,
        random_seed=i,
        honest=True,
    )
    model.fit(train_ds)
    
    sub_predictions = model.predict(serving_ds, verbose=0)[:,0]
    if predictions is None:
        predictions = sub_predictions
    else:
        predictions += sub_predictions
    num_predictions += 1

predictions/=num_predictions

kaggle_predictions = pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (predictions >= 0.5).astype(int)
    })

make_submission(kaggle_predictions)

i:0


[INFO 2026-09-06T06:35:26.676299474+00:00 kernel.cc:1214] Loading model from path /tmp/tmpfy7je7o8/model/ with prefix d49137c753c64af6
[INFO 2026-09-06T06:35:26.680145042+00:00 kernel.cc:1046] Use fast generic engine


i:1


[INFO 2026-09-06T06:35:27.461442843+00:00 kernel.cc:1214] Loading model from path /tmp/tmptlt_k5st/model/ with prefix 9055314bcbc643be
[INFO 2026-09-06T06:35:27.467646819+00:00 kernel.cc:1046] Use fast generic engine


i:2


...

i:98


[INFO 2026-09-06T06:36:51.182642705+00:00 kernel.cc:1214] Loading model from path /tmp/tmpz8a5tvqo/model/ with prefix d4f39a560cd94df9
[INFO 2026-09-06T06:36:51.187194489+00:00 kernel.cc:1046] Use fast generic engine


i:99


[INFO 2026-09-06T06:36:52.096324413+00:00 kernel.cc:1214] Loading model from path /tmp/tmpje5okqbz/model/ with prefix 062401ee2ac2420d
[INFO 2026-09-06T06:36:52.106334087+00:00 kernel.cc:1046] Use fast generic engine


Submission exported to /kaggle/working/submission.csv
